# 🧠 NLP → Agentic AI: Consultant Training Notebook
**Phases covered:** Fundamentals · spaCy · BERT/RoBERTa · OCR · Pipelines · Agentic AI  
**Seed data:** Auto-generated via multi-provider LLM — **no GPU required**  
**Estimated time:** 12 weeks of hands-on practice  

---
## 🔌 LLM Provider Priority Chain (All CPU-friendly)

| Priority | Provider | Key / Config | Model | GPU? |
|----------|----------|-------------|-------|------|
| 1 | **Anthropic Claude** | `ANTHROPIC_API_KEY` | claude-sonnet-4-20250514 | ☁️ Cloud |
| 2 | **Groq** (free tier) | `GROQ_API_KEY` | llama3-70b / mixtral-8x7b | ☁️ Cloud |
| 3 | **Azure OpenAI** | `AZURE_OPENAI_KEY` + endpoint | gpt-4o | ☁️ Cloud |
| 4 | **Ollama (CPU)** | Ollama on localhost | phi3 / gemma2:2b | 💻 CPU |
| 5 | **HuggingFace CPU** | No key — GGUF via llama-cpp | Phi-3-mini / Qwen2-1.5B | 💻 CPU |
| 6 | **Static fallback** | Nothing needed | Pre-built data | ✅ Always |

---
> **No GPU? No problem.** Providers 1–3 are cloud APIs (no local compute).
> Provider 4 (Ollama) runs tiny models (phi3, gemma2:2b) on as little as 4GB RAM.
> Provider 5 uses GGUF quantized models via llama-cpp-python — pure CPU, ~2-4GB RAM.

---
> **How to use this notebook:**  
> 1. Set whichever keys you have in **Cell 2 (Config)**  
> 2. Run **Cell 3** — it auto-picks the best available provider  
> 3. Run **Cell 4** — unified `llm_generate()` ready  
> 4. Run **Cell 5** — seed data generated  
> 5. Work through Phase 1–6 sequentially

## ⚙️ Cell 1: Install All Dependencies
Run once per environment

In [ ]:
import subprocess, sys

packages = [
    # ── LLM Providers (cloud) ───────────────────────────────
    'anthropic',              # Provider 1: Claude
    'groq',                   # Provider 2: Groq (free, fast)
    'openai',                 # Provider 3: Azure OpenAI
    'requests',               # Provider 4: Ollama REST

    # ── LLM local CPU (no GPU) ──────────────────────────────
    'llama-cpp-python',       # Provider 5: GGUF models on CPU
    'huggingface_hub',        # Download GGUF model files

    # ── NLP Core ────────────────────────────────────────────
    'spacy', 'transformers', 'datasets',
    'torch',                  # CPU-only torch is fine for inference
    'sentence-transformers',
    'faiss-cpu',
    'nltk', 'scikit-learn',
    'pandas', 'numpy', 'tqdm',

    # ── OCR ─────────────────────────────────────────────────
    'paddlepaddle', 'paddleocr',  # CPU version by default
    'boto3',
    'Pillow',

    # ── Agent Frameworks ────────────────────────────────────
    'langchain', 'langchain-anthropic', 'langchain-openai',
]

for pkg in packages:
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
    except Exception as e:
        print(f'  ⚠️  {pkg} failed (may be optional): {e}')

subprocess.check_call([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'])

import torch
print('✅ All dependencies installed!')
print(f'   PyTorch: {torch.__version__}')
print(f'   GPU available: {torch.cuda.is_available()} (CPU-only is fine for this training)')


## 🔑 Cell 2: Configuration — Set Your LLM Keys
Fill in whichever provider(s) you have access to. The notebook will auto-pick the best available.

In [9]:
import os, json
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# ══════════════════════════════════════════════════════════════
#  SET YOUR KEYS — leave empty string '' if you don't have it
#  You only need ONE provider. Cloud APIs need no local compute.
# ══════════════════════════════════════════════════════════════

# ── Provider 1: Anthropic Claude ──────────────────────────────
ANTHROPIC_API_KEY      = os.environ.get('ANTHROPIC_API_KEY', '')

# ── Provider 2: Groq (FREE — https://console.groq.com) ────────
#   Sign up free, get key instantly at https://console.groq.com
GROQ_API_KEY           = os.environ.get('GROQ_API_KEY', '')

# ── Provider 3: OpenAI (PAID) ─────────────────────────────────
OPENAI_API_KEY         = os.environ.get('OPENAI_API_KEY', '')

# ── Provider 4: Azure OpenAI ──────────────────────────────────
AZURE_OPENAI_KEY       = os.environ.get('AZURE_OPENAI_KEY', '')
AZURE_OPENAI_ENDPOINT  = os.environ.get('AZURE_OPENAI_ENDPOINT', '')
AZURE_DEPLOYMENT_NAME  = os.environ.get('AZURE_DEPLOYMENT_NAME', 'gpt-4o')

# ── Model Selection ───────────────────────────────────────────
ANTHROPIC_MODEL        = 'claude-sonnet-4-20250514'
OPENAI_MODEL           = 'gpt-4o'
GROQ_MODEL             = 'llama-3.3-70b-versatile'

# ── Local Models (Ollama) ─────────────────────────────────────
OLLAMA_HOST            = os.environ.get('OLLAMA_HOST', 'http://localhost:11434')
OLLAMA_MODEL           = os.environ.get('OLLAMA_MODEL', 'phi3')  # Best for CPU

# ═══════════════════════════════════════════════════════════════
print('✅ API Key Status:')
print(f'  Claude key     : {"SET" if ANTHROPIC_API_KEY else "not set"}')
print(f'  OpenAI key     : {"SET" if OPENAI_API_KEY else "not set"}')
print(f'  Groq key       : {"SET" if GROQ_API_KEY else "not set"}')
print(f'  Azure key      : {"SET" if AZURE_OPENAI_KEY else "not set"}')
print(f'  Ollama (local) : {OLLAMA_HOST} - {OLLAMA_MODEL}')


✅ API Key Status:
  Claude key     : SET
  OpenAI key     : not set
  Groq key       : SET
  Azure key      : not set
  Ollama (local) : http://localhost:11434 - phi3


## 🔍 Cell 3: Auto-Detect Best LLM Provider
Runs a quick check on each provider in priority order and selects the first that responds.

In [10]:
import requests

ACTIVE_PROVIDER = None

# ─── Check Provider 1: Anthropic Claude ─────────────────────
def check_claude():
    if not ANTHROPIC_API_KEY: return False
    try:
        import anthropic
        c = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        c.messages.create(model='claude-sonnet-4-20250514', max_tokens=5,
                          messages=[{'role':'user','content':'hi'}])
        return True
    except Exception as e:
        print(f'    Claude: {e}'); return False

# ─── Check Provider 2: Groq (free) ──────────────────────────
def check_groq():
    if not GROQ_API_KEY: return False
    try:
        from groq import Groq
        c = Groq(api_key=GROQ_API_KEY)
        c.chat.completions.create(model=GROQ_MODEL, max_tokens=5,
                                  messages=[{'role':'user','content':'hi'}])
        return True
    except Exception as e:
        print(f'    Groq: {e}'); return False

# ─── Check Provider 3: Azure OpenAI ─────────────────────────
def check_azure():
    if not AZURE_OPENAI_KEY or not AZURE_OPENAI_ENDPOINT: return False
    try:
        from openai import AzureOpenAI
        c = AzureOpenAI(api_key=AZURE_OPENAI_KEY,
                        azure_endpoint=AZURE_OPENAI_ENDPOINT,
                        api_version=AZURE_API_VERSION)
        c.chat.completions.create(model=AZURE_DEPLOYMENT_NAME, max_tokens=5,
                                  messages=[{'role':'user','content':'hi'}])
        return True
    except Exception as e:
        print(f'    Azure: {e}'); return False

# ─── Check Provider 4: Ollama (CPU-friendly) ─────────────────
def check_ollama():
    try:
        r = requests.get(f'{OLLAMA_HOST}/api/tags', timeout=3)
        if r.status_code != 200: return False
        models = [m['name'].split(':')[0] for m in r.json().get('models', [])]
        print(f'    Ollama models found: {models}')
        return len(models) > 0
    except Exception as e:
        print(f'    Ollama: {e}'); return False

# ─── Check Provider 5: GGUF CPU (llama-cpp-python) ───────────
def check_gguf_cpu():
    try:
        import llama_cpp   # Will import fail if not installed
        from huggingface_hub import hf_hub_download
        return True        # Available — model downloads on first use
    except ImportError as e:
        print(f'    GGUF CPU: {e}'); return False

# ─── Auto-detect ─────────────────────────────────────────────
print('🔍 Detecting available LLM providers (no GPU required)...')
print()

PROVIDER_CHAIN = [
    ('claude',    'Anthropic Claude  (cloud)',       check_claude),
    ('groq',      'Groq  (free cloud, fast)',        check_groq),
    ('azure',     'Azure OpenAI  (cloud)',           check_azure),
    ('ollama',    'Ollama  (local CPU)',              check_ollama),
    ('gguf_cpu',  'HuggingFace GGUF  (local CPU)',   check_gguf_cpu),
    ('fallback',  'Static fallback  (no LLM)',       lambda: True),
]

for provider_id, display_name, check_fn in PROVIDER_CHAIN:
    print(f'  Checking {display_name}...')
    if check_fn():
        ACTIVE_PROVIDER = provider_id
        print(f'\n✅ Selected: {display_name}  [{provider_id}]')
        break

if ACTIVE_PROVIDER == 'fallback':
    print('⚠️  No LLM found — will use static seed data')
    print('   💡 Tip: Get a free Groq key at https://console.groq.com (takes 2 minutes)')


🔍 Detecting available LLM providers (no GPU required)...

  Checking Anthropic Claude  (cloud)...
    Claude: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'Your credit balance is too low to access the Anthropic API. Please go to Plans & Billing to upgrade or purchase credits.'}, 'request_id': 'req_011Ca3PqXGbQV8iFoY3R5Nxt'}
  Checking Groq  (free cloud, fast)...

✅ Selected: Groq  (free cloud, fast)  [groq]


## 🤖 Cell 4: Unified LLM Interface
Single `llm_generate()` function that works identically across all providers.

In [11]:
# ══════════════════════════════════════════════════════════════
#  UNIFIED LLM INTERFACE — CPU-friendly, no GPU required
# ══════════════════════════════════════════════════════════════
_gguf_model_instance = None   # Cache loaded GGUF model

def llm_generate(prompt: str, max_tokens: int = 2000) -> str:
    """Route to active provider. No GPU required for any option."""
    global _gguf_model_instance

    # ── Provider 1: Claude ────────────────────────────────────
    if ACTIVE_PROVIDER == 'claude':
        import anthropic
        c = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        return c.messages.create(
            model='claude-sonnet-4-20250514', max_tokens=max_tokens,
            messages=[{'role': 'user', 'content': prompt}]
        ).content[0].text

    # ── Provider 2: Groq (free cloud — fast inference) ────────
    elif ACTIVE_PROVIDER == 'groq':
        from groq import Groq
        c = Groq(api_key=GROQ_API_KEY)
        return c.chat.completions.create(
            model=GROQ_MODEL, max_tokens=max_tokens,
            messages=[
                {'role': 'system', 'content': 'Return only valid JSON arrays. No markdown.'},
                {'role': 'user', 'content': prompt}
            ]
        ).choices[0].message.content

    # ── Provider 3: Azure OpenAI ──────────────────────────────
    elif ACTIVE_PROVIDER == 'azure':
        from openai import AzureOpenAI
        c = AzureOpenAI(api_key=AZURE_OPENAI_KEY,
                        azure_endpoint=AZURE_OPENAI_ENDPOINT,
                        api_version=AZURE_API_VERSION)
        return c.chat.completions.create(
            model=AZURE_DEPLOYMENT_NAME, max_tokens=max_tokens,
            messages=[
                {'role': 'system', 'content': 'Return only valid JSON. No markdown.'},
                {'role': 'user', 'content': prompt}
            ]
        ).choices[0].message.content

    # ── Provider 4: Ollama (CPU local — tiny models OK) ───────
    elif ACTIVE_PROVIDER == 'ollama':
        import requests
        # Pick best available model (prefer phi3 for CPU)
        tags = requests.get(f'{OLLAMA_HOST}/api/tags').json()
        available = [m['name'] for m in tags.get('models', [])]
        cpu_preferred = ['phi3', 'phi3.5', 'gemma2:2b', 'qwen2:1.5b',
                         'tinyllama', 'mistral', 'llama3']
        model_to_use = next(
            (m for pref in cpu_preferred for m in available if pref in m),
            available[0] if available else OLLAMA_MODEL
        )
        print(f'  Using Ollama model: {model_to_use}')
        r = requests.post(f'{OLLAMA_HOST}/api/generate', json={
            'model': model_to_use,
            'prompt': f'[INST] {prompt} [/INST]',
            'stream': False,
            'options': {'num_predict': max_tokens, 'temperature': 0.7,
                        'num_ctx': 4096}    # Context window
        })
        return r.json()['response']

    # ── Provider 5: GGUF CPU (llama-cpp-python) ───────────────
    #    Downloads GGUF model once, caches for session.
    #    Phi-3-mini-q4: 2.2GB RAM, ~5-10 tokens/sec on modern CPU
    elif ACTIVE_PROVIDER == 'gguf_cpu':
        from llama_cpp import Llama
        from huggingface_hub import hf_hub_download

        if _gguf_model_instance is None:
            print(f'  📥 Downloading {GGUF_FILE} (first run only)...')
            model_path = hf_hub_download(
                repo_id=GGUF_REPO,
                filename=GGUF_FILE,
                local_dir='./models'
            )
            print(f'  🔄 Loading model (CPU)...')
            _gguf_model_instance = Llama(
                model_path=model_path,
                n_ctx=4096,           # Context window
                n_threads=4,          # CPU threads (adjust to your core count)
                n_gpu_layers=0,       # 0 = pure CPU, no GPU needed
                verbose=False
            )
            print('  ✅ GGUF model loaded on CPU')

        output = _gguf_model_instance(
            f'<|user|>\n{prompt}<|end|>\n<|assistant|>',
            max_tokens=max_tokens,
            temperature=0.7,
            stop=['<|end|>', '<|user|>']
        )
        return output['choices'][0]['text']

    return '[]'   # Fallback — static seed data used


def parse_llm_json(raw: str) -> list:
    """Safely parse JSON from any LLM output, stripping code fences."""
    raw = raw.strip()
    for fence in ['```json', '```JSON', '```']:
        raw = raw.replace(fence, '')
    raw = raw.strip()
    start, end = raw.find('['), raw.rfind(']')
    if start != -1 and end != -1:
        raw = raw[start:end+1]
    return json.loads(raw)


print(f'✅ LLM interface ready — provider: [{ACTIVE_PROVIDER}] — GPU required: NO')
print()
# Smoke test
if ACTIVE_PROVIDER != 'fallback':
    try:
        test = llm_generate('Return this exact JSON with no other text: [{"ok": true}]',
                             max_tokens=30)
        result = parse_llm_json(test)
        print(f'  Smoke test ✅ → {result}')
    except Exception as e:
        print(f'  Smoke test failed: {e}')
        print('  Switching to static fallback...')
        ACTIVE_PROVIDER = 'fallback'


✅ LLM interface ready — provider: [groq] — GPU required: NO

  Smoke test ✅ → [{'ok': True}]


## 🌱 Cell 5: Generate Seed Data
Generates realistic NLP training documents via the active LLM provider.

In [22]:
# ══════════════════════════════════════════════════════════════
#  GENERATE SEED DATA via active LLM provider
# ══════════════════════════════════════════════════════════════

SEED_PROMPT_TEMPLATE = """
Generate {count} realistic {domain} text samples for NLP training (Indian business context).
Include named entities: ORG, PERSON, DATE, MONEY, LOC, GPE.

Return ONLY a valid JSON array. Each element must have:
- "text": document text (2-5 sentences, realistic, varied)
- "label": category (invoice/contract/complaint/medical/news)
- "entities": [{{"text": str, "label": str, "start": int, "end": int}}]
- "metadata": {{"domain": "{domain}", "language": "en", "difficulty": "medium"}}

No markdown. No preamble. Return only the JSON array.
"""

def generate_seed_data(domain: str, count: int = 5) -> list:
    """Generate seed docs via the active LLM provider."""
    prompt = SEED_PROMPT_TEMPLATE.format(domain=domain, count=count)
    try:
        raw = llm_generate(prompt, max_tokens=2000)
        docs = parse_llm_json(raw)
        if not isinstance(docs, list) or len(docs) == 0:
            raise ValueError('Empty or non-list response')
        return docs
    except Exception as e:
        print(f'  ⚠️  Generation failed for [{domain}]: {e}')
        return []  # Caller will fall back to static data


domains = [
    'financial invoices',
    'legal contracts and NDAs',
    'customer support complaint emails',
    'medical discharge summaries',
    'financial news articles'
]

SEED_DATA = {}
print(f'🔄 Generating seed data via [{ACTIVE_PROVIDER}]...')

for domain in domains:
    key = domain.replace(' ', '_').replace('/', '_')
    docs = generate_seed_data(domain, count=4)
    if docs:
        SEED_DATA[key] = docs
        print(f'  ✅ {domain}: {len(docs)} docs')
    else:
        print(f'  ⚠️  {domain}: generation failed, will use fallback')

ALL_DOCS = [doc for docs in SEED_DATA.values() for doc in docs]
print(f'\n📄 LLM-generated docs: {len(ALL_DOCS)}')


🔄 Generating seed data via [groq]...
  ✅ financial invoices: 4 docs
  ✅ legal contracts and NDAs: 4 docs
  ✅ customer support complaint emails: 4 docs
  ✅ medical discharge summaries: 4 docs
  ✅ financial news articles: 4 docs

📄 LLM-generated docs: 20


In [23]:
# ── FALLBACK: Pre-generated seed data (use if API is unavailable) ──
FALLBACK_SEED = [
    {
        'text': 'Acme Corp issued invoice #INV-2024-0451 to GlobalTech Ltd on March 15, 2024 for $12,450.00. Payment is due within 30 days at our Mumbai office.',
        'label': 'invoice',
        'entities': [
            {'text': 'Acme Corp', 'label': 'ORG', 'start': 0, 'end': 9},
            {'text': 'GlobalTech Ltd', 'label': 'ORG', 'start': 38, 'end': 52},
            {'text': 'March 15, 2024', 'label': 'DATE', 'start': 56, 'end': 70},
            {'text': '$12,450.00', 'label': 'MONEY', 'start': 75, 'end': 85},
            {'text': 'Mumbai', 'label': 'GPE', 'start': 126, 'end': 132},
        ],
        'metadata': {'domain': 'financial_invoices', 'language': 'en', 'difficulty': 'medium'}
    },
    {
        'text': 'This Service Agreement is entered into between Infosys Limited and Dr. Priya Sharma effective January 1, 2024. The contract covers software development services in Bengaluru for a period of 12 months.',
        'label': 'contract',
        'entities': [
            {'text': 'Infosys Limited', 'label': 'ORG', 'start': 47, 'end': 62},
            {'text': 'Dr. Priya Sharma', 'label': 'PERSON', 'start': 67, 'end': 83},
            {'text': 'January 1, 2024', 'label': 'DATE', 'start': 94, 'end': 109},
            {'text': 'Bengaluru', 'label': 'GPE', 'start': 163, 'end': 172},
        ],
        'metadata': {'domain': 'legal_contracts', 'language': 'en', 'difficulty': 'medium'}
    },
    {
        'text': 'Dear Support Team, I ordered a Dell XPS laptop from your New Delhi store on April 2, 2024 but received a damaged unit. Please arrange a replacement or refund of Rs. 89,000 urgently.',
        'label': 'complaint',
        'entities': [
            {'text': 'Dell XPS', 'label': 'ORG', 'start': 34, 'end': 42},
            {'text': 'New Delhi', 'label': 'GPE', 'start': 57, 'end': 66},
            {'text': 'April 2, 2024', 'label': 'DATE', 'start': 74, 'end': 87},
            {'text': 'Rs. 89,000', 'label': 'MONEY', 'start': 158, 'end': 168},
        ],
        'metadata': {'domain': 'customer_support_emails', 'language': 'en', 'difficulty': 'easy'}
    },
    {
        'text': 'Patient Mr. Rajesh Kumar was admitted to Apollo Hospital, Chennai on 12 February 2024 with acute chest pain. He was discharged on 18 February 2024 after successful angioplasty.',
        'label': 'medical_note',
        'entities': [
            {'text': 'Mr. Rajesh Kumar', 'label': 'PERSON', 'start': 8, 'end': 24},
            {'text': 'Apollo Hospital', 'label': 'ORG', 'start': 42, 'end': 57},
            {'text': 'Chennai', 'label': 'GPE', 'start': 59, 'end': 66},
            {'text': '12 February 2024', 'label': 'DATE', 'start': 70, 'end': 86},
            {'text': '18 February 2024', 'label': 'DATE', 'start': 128, 'end': 144},
        ],
        'metadata': {'domain': 'medical_discharge_summaries', 'language': 'en', 'difficulty': 'hard'}
    },
    {
        'text': 'TCS reported a net profit of Rs. 11,058 crore for Q3 FY2024, up 2.4% year-on-year. CEO K. Krithivasan commented that growth was driven by BFSI and life sciences verticals in North America and Europe.',
        'label': 'news',
        'entities': [
            {'text': 'TCS', 'label': 'ORG', 'start': 0, 'end': 3},
            {'text': 'Rs. 11,058 crore', 'label': 'MONEY', 'start': 30, 'end': 46},
            {'text': 'Q3 FY2024', 'label': 'DATE', 'start': 51, 'end': 60},
            {'text': 'K. Krithivasan', 'label': 'PERSON', 'start': 79, 'end': 93},
            {'text': 'North America', 'label': 'LOC', 'start': 172, 'end': 185},
            {'text': 'Europe', 'label': 'LOC', 'start': 190, 'end': 196},
        ],
        'metadata': {'domain': 'financial_news', 'language': 'en', 'difficulty': 'hard'}
    }
]

# Use fallback if needed
if 'ALL_DOCS' not in dir() or len(ALL_DOCS) == 0:
    ALL_DOCS = FALLBACK_SEED
    print('Using fallback seed data')
else:
    print(f'Using API-generated seed data ({len(ALL_DOCS)} docs)')

Using API-generated seed data (20 docs)


---
# 📗 PHASE 1: NLP Fundamentals
**Goal:** Understand text representation, tokenization, POS, NER, and text cleaning pipelines  
**Duration:** Week 1–2

In [24]:
# ─── 1.1 Tokenization ───────────────────────────────────────
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import stopwords
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

sample_text = ALL_DOCS[0]['text']
print('📄 Sample text:')
print(sample_text)
print()

# Sentence tokenization
sentences = sent_tokenize(sample_text)
print(f'📌 Sentences ({len(sentences)}):')
for i, s in enumerate(sentences):
    print(f'  [{i}] {s}')

# Word tokenization
words = word_tokenize(sample_text)
print(f'\n📌 Tokens ({len(words)}): {words[:12]} ...')

# Stopword removal
stop_words = set(stopwords.words('english'))
filtered = [w for w in words if w.lower() not in stop_words and w.isalpha()]
print(f'\n📌 After stopword removal ({len(filtered)}): {filtered[:10]} ...')

# Stemming vs Lemmatization
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

test_words = ['running', 'payments', 'issued', 'development', 'companies']
print('\n📌 Stemming vs Lemmatization:')
print(f"{'Word':<15} {'Stem':<15} {'Lemma':<15}")
print('-' * 45)
for w in test_words:
    print(f"{w:<15} {stemmer.stem(w):<15} {lemmatizer.lemmatize(w):<15}")

📄 Sample text:
Invoice No. INV001 dated 2022-02-15 for Rs. 10,000 issued by Reliance Industries to Mr. Ramesh Kumar for the supply of goods from Mumbai to Delhi. Payment is due on 2022-03-15. Please find the details of the transaction below.



LookupError: 
**********************************************************************
  Resource 'punkt_tab' not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('punkt_tab')

  For more information see: https://www.nltk.org/data.html

  Attempted to load 'tokenizers/punkt_tab/english/'

  Searched in:
    - '/Users/kevin/nltk_data'
    - '/Users/kevin/Programming/Mahendra-Notebooks/.venv/nltk_data'
    - '/Users/kevin/Programming/Mahendra-Notebooks/.venv/share/nltk_data'
    - '/Users/kevin/Programming/Mahendra-Notebooks/.venv/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************


In [ ]:
# ─── 1.2 POS Tagging & Basic Text Cleaning ──────────────────
import re

def clean_text(text: str) -> str:
    """Standard NLP preprocessing pipeline"""
    text = text.lower()                          # lowercase
    text = re.sub(r'http\S+', '', text)          # remove URLs
    text = re.sub(r'[^\w\s\.\,\$\#\@]', '', text) # keep useful punctuation
    text = re.sub(r'\s+', ' ', text).strip()     # normalize whitespace
    return text

# Apply to seed data
for doc in ALL_DOCS[:3]:
    cleaned = clean_text(doc['text'])
    print(f"Label : {doc['label']}")
    print(f"Before: {doc['text'][:80]}...")
    print(f"After : {cleaned[:80]}...")
    print()

In [ ]:
# ─── 1.3 TF-IDF Vectorization ───────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

corpus = [doc['text'] for doc in ALL_DOCS]
labels = [doc['label'] for doc in ALL_DOCS]

vectorizer = TfidfVectorizer(max_features=50, stop_words='english')
X = vectorizer.fit_transform(corpus)

print(f'TF-IDF matrix shape: {X.shape}')
print(f'Vocabulary size: {len(vectorizer.vocabulary_)}')
print()

# Top terms per document
feature_names = vectorizer.get_feature_names_out()
print('Top 5 TF-IDF terms per document:')
for i, (doc, label) in enumerate(zip(ALL_DOCS[:5], labels[:5])):
    row = X[i].toarray()[0]
    top_idx = np.argsort(row)[::-1][:5]
    top_terms = [(feature_names[j], round(row[j], 3)) for j in top_idx if row[j] > 0]
    print(f"  [{label}] → {top_terms}")

---
# 📘 PHASE 2: spaCy NLP Pipeline
**Goal:** Build production spaCy pipelines — custom NER, entity rulers, and pipeline components  
**Duration:** Week 3–4

In [ ]:
# ─── 2.1 spaCy Core Pipeline ────────────────────────────────
import spacy
from spacy import displacy

nlp = spacy.load('en_core_web_sm')
print('Pipeline components:', nlp.pipe_names)
print()

# Process seed documents
for doc_data in ALL_DOCS[:3]:
    doc = nlp(doc_data['text'])
    print(f"🏷️  Label: {doc_data['label']}")
    print(f"   Text: {doc_data['text'][:70]}...")
    print('   Entities found by spaCy:')
    for ent in doc.ents:
        print(f"     {ent.text:<25} {ent.label_:<12} {spacy.explain(ent.label_)}")
    print()

In [ ]:
# ─── 2.2 EntityRuler — Domain-Specific Pattern Matching ──────
from spacy.pipeline import EntityRuler

# Create a fresh pipeline with EntityRuler before NER
nlp_custom = spacy.load('en_core_web_sm', disable=['ner'])
ruler = nlp_custom.add_pipe('entity_ruler')

# Domain-specific patterns (Indian business context)
patterns = [
    # Invoice numbers
    {'label': 'INVOICE_ID', 'pattern': [{'TEXT': {'REGEX': 'INV-\\d{4}-\\d{4}'}}]},
    # Indian currency
    {'label': 'MONEY_INR', 'pattern': [{'TEXT': 'Rs.'}, {'LIKE_NUM': True}]},
    {'label': 'MONEY_INR', 'pattern': [{'TEXT': {'REGEX': 'Rs\\.\\s*[\\d,]+'}  }]},
    # GST numbers
    {'label': 'GST_NO', 'pattern': [{'TEXT': {'REGEX': '[0-9]{2}[A-Z]{5}[0-9]{4}[A-Z]{1}[1-9A-Z]{1}Z[0-9A-Z]{1}'}}]},
    # Indian cities
    {'label': 'CITY_IN', 'pattern': [{'LOWER': {'IN': ['mumbai', 'delhi', 'bengaluru', 'chennai', 'hyderabad', 'pune', 'kolkata']}}]},
    # Document types
    {'label': 'DOC_TYPE', 'pattern': [{'LOWER': {'IN': ['invoice', 'purchase order', 'contract', 'agreement', 'nda']}}]},
    # Quarter references
    {'label': 'FISCAL_PERIOD', 'pattern': [{'TEXT': {'REGEX': 'Q[1-4]\\s+FY\\d{4}'}}]},
]

ruler.add_patterns(patterns)

# Test on seed data
test_texts = [
    'Invoice INV-2024-0451 for Rs. 89,000 was sent to the Mumbai office.',
    'GST registration 27AABCU9603R1ZX for Q3 FY2024 filing is pending.',
    'The service agreement and NDA were signed in Bengaluru.',
]

print('Custom EntityRuler Results:')
print('=' * 60)
for text in test_texts:
    doc = nlp_custom(text)
    print(f'\nText: {text}')
    print('Entities:')
    for ent in doc.ents:
        print(f'  {ent.text:<30} → {ent.label_}')

In [ ]:
# ─── 2.3 Custom Pipeline Component ──────────────────────────
from spacy.language import Language
from spacy.tokens import Doc, Span

# Custom component: Document Classifier
@Language.component('document_classifier')
def document_classifier(doc):
    """Classify document type based on keyword signals"""
    text_lower = doc.text.lower()

    rules = {
        'invoice':   ['invoice', 'inv-', 'payment due', 'amount payable', 'bill to'],
        'contract':  ['agreement', 'hereby', 'terms and conditions', 'clause', 'signed'],
        'complaint': ['complaint', 'refund', 'damaged', 'issue', 'disappointed', 'urgent'],
        'medical':   ['patient', 'hospital', 'admitted', 'discharged', 'diagnosis'],
        'news':      ['reported', 'quarter', 'profit', 'growth', 'announced'],
    }

    scores = {label: sum(1 for kw in kws if kw in text_lower) for label, kws in rules.items()}
    best_label = max(scores, key=scores.get)

    # Store on Doc using extension
    if not Doc.has_extension('doc_type'):
        Doc.set_extension('doc_type', default='unknown')
    if not Doc.has_extension('doc_score'):
        Doc.set_extension('doc_score', default=0)

    doc._.doc_type = best_label if scores[best_label] > 0 else 'unknown'
    doc._.doc_score = scores[best_label]
    return doc

# Add component to pipeline
nlp2 = spacy.load('en_core_web_sm')
nlp2.add_pipe('document_classifier', last=True)

print('Pipeline:', nlp2.pipe_names)
print()
print('Document Classification Results:')
print('=' * 70)

for doc_data in ALL_DOCS[:5]:
    doc = nlp2(doc_data['text'])
    correct = '✅' if doc._.doc_type == doc_data['label'] else '❌'
    print(f"{correct} Predicted: {doc._.doc_type:<15} Actual: {doc_data['label']:<15} Score: {doc._.doc_score}")

In [ ]:
# ─── 2.4 NER Evaluation Against Ground Truth ────────────────
from sklearn.metrics import classification_report

def evaluate_ner(docs_with_gt: list, nlp_model) -> dict:
    """
    Compare spaCy NER predictions vs ground truth entities.
    Returns precision, recall, F1 per entity type.
    """
    y_true, y_pred = [], []

    for doc_data in docs_with_gt:
        text = doc_data['text']
        gt_entities = {(e['start'], e['end'], e['label']) for e in doc_data.get('entities', [])}

        doc = nlp_model(text)
        pred_entities = {(ent.start_char, ent.end_char, ent.label_) for ent in doc.ents}

        # Token-level approximation for report
        for entity in gt_entities:
            y_true.append(entity[2])
            y_pred.append(entity[2] if entity in pred_entities else 'MISSED')

        for entity in pred_entities - gt_entities:
            y_true.append('FALSE_POS')
            y_pred.append(entity[2])

    return y_true, y_pred

nlp_eval = spacy.load('en_core_web_sm')
y_true, y_pred = evaluate_ner(ALL_DOCS, nlp_eval)

print('NER Evaluation (spaCy en_core_web_sm vs Ground Truth):')
print('=' * 60)
print(classification_report(y_true, y_pred, zero_division=0))

# Entity counts
print('\nGround Truth Entity Distribution:')
from collections import Counter
all_ents = [e['label'] for doc in ALL_DOCS for e in doc.get('entities', [])]
for label, count in Counter(all_ents).most_common():
    print(f'  {label:<12} {count}')

---
# 📙 PHASE 3: BERT & RoBERTa
**Goal:** Fine-tune BERT for text classification and NER; understand RoBERTa/DistilBERT trade-offs  
**Duration:** Week 5–6

In [ ]:
# ─── 3.1 BERT Tokenizer Deep Dive ───────────────────────────
from transformers import BertTokenizer, RobertaTokenizer, AutoTokenizer

bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
roberta_tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

sample = ALL_DOCS[0]['text']
print('Input text:')
print(sample)
print()

# BERT tokenization
bert_tokens = bert_tokenizer.tokenize(sample)
bert_ids = bert_tokenizer.encode(sample, add_special_tokens=True)
print(f'BERT tokens ({len(bert_tokens)}): {bert_tokens[:15]} ...')
print(f'BERT IDs  ({len(bert_ids)}): {bert_ids[:10]} ...')
print(f'[CLS] token ID: {bert_tokenizer.cls_token_id}')
print(f'[SEP] token ID: {bert_tokenizer.sep_token_id}')
print()

# RoBERTa tokenization (BPE, not WordPiece)
roberta_tokens = roberta_tokenizer.tokenize(sample)
print(f'RoBERTa tokens ({len(roberta_tokens)}): {roberta_tokens[:15]} ...')
print()

# Batch encoding with padding & attention mask
batch_texts = [doc['text'] for doc in ALL_DOCS[:4]]
encoded = bert_tokenizer(
    batch_texts,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors='pt'
)
print('Batch encoding shapes:')
print(f'  input_ids:      {encoded["input_ids"].shape}')
print(f'  attention_mask: {encoded["attention_mask"].shape}')

In [ ]:
# ─── 3.2 BERT Text Classification (Fine-tuning Setup) ───────
import torch
from transformers import (
    BertForSequenceClassification,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    TrainingArguments,
    Trainer
)
from torch.utils.data import Dataset

# Label mapping from seed data
LABELS = sorted(set(doc['label'] for doc in ALL_DOCS))
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
print('Label mapping:', label2id)

# Custom Dataset
class DocumentDataset(Dataset):
    def __init__(self, docs, tokenizer, max_len=128):
        self.docs = docs
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.docs)

    def __getitem__(self, idx):
        doc = self.docs[idx]
        encoding = self.tokenizer(
            doc['text'],
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(label2id[doc['label']], dtype=torch.long)
        }

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')
model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id
)

print(f'\n✅ Model loaded: distilbert-base-uncased')
print(f'   Parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'   Labels: {LABELS}')

dataset = DocumentDataset(ALL_DOCS, tokenizer)
sample_batch = dataset[0]
print(f'\nSample batch keys: {list(sample_batch.keys())}')
print(f'input_ids shape: {sample_batch["input_ids"].shape}')

In [ ]:
# ─── 3.3 Zero-shot Classification with Pretrained Models ────
from transformers import pipeline

# Zero-shot classifier (no fine-tuning needed)
classifier = pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli'
)

candidate_labels = ['invoice', 'legal contract', 'customer complaint', 'medical document', 'financial news']

print('Zero-shot Classification Results:')
print('=' * 70)

for doc_data in ALL_DOCS[:5]:
    result = classifier(doc_data['text'][:512], candidate_labels)
    top_pred = result['labels'][0]
    top_score = result['scores'][0]
    print(f"Actual : {doc_data['label']}")
    print(f"Predicted: {top_pred} ({top_score:.2%})")
    print(f"All scores: {dict(zip(result['labels'], [f'{s:.2%}' for s in result['scores']]))}")
    print()

In [ ]:
# ─── 3.4 Model Comparison: BERT vs RoBERTa vs DistilBERT ────
import time

models_to_compare = [
    ('distilbert-base-uncased', 'DistilBERT', '66M params, 40% smaller than BERT'),
    ('bert-base-uncased',       'BERT',       '110M params, WordPiece, bidirectional'),
    ('roberta-base',            'RoBERTa',    '125M params, BPE, no NSP, more data'),
]

test_text = ALL_DOCS[0]['text']

print('Model Comparison — Tokenization Speed & Token Count:')
print('=' * 70)
print(f"{'Model':<15} {'Tokens':<10} {'Time(ms)':<12} {'Notes'}")
print('-' * 70)

for model_name, display_name, notes in models_to_compare:
    tok = AutoTokenizer.from_pretrained(model_name)
    start = time.time()
    for _ in range(100):
        tokens = tok.tokenize(test_text)
    elapsed = (time.time() - start) * 10  # per call in ms
    print(f"{display_name:<15} {len(tokens):<10} {elapsed:<12.2f} {notes}")

print()
print('When to choose which model:')
print('  DistilBERT → production inference, latency-sensitive, limited compute')
print('  BERT       → balanced tasks, well-documented, many fine-tuned variants')
print('  RoBERTa    → higher accuracy tasks, classification, QA, NER at scale')
print('  DeBERTa    → leaderboard-quality NLU, complex reasoning tasks')

---
# 📕 PHASE 4: OCR — PaddleOCR & AWS Textract
**Goal:** Extract text from images and PDFs; integrate OCR output into NLP pipelines  
**Duration:** Week 7–8

In [ ]:
# ─── 4.1 PaddleOCR — Basic Inference ────────────────────────
# NOTE: Requires PaddleOCR installation. GPU strongly recommended for prod.

try:
    from paddleocr import PaddleOCR
    from PIL import Image, ImageDraw, ImageFont
    import numpy as np
    import urllib.request

    # Initialize PaddleOCR (English + Hindi multilingual)
    ocr = PaddleOCR(
        use_angle_cls=True,   # Auto-rotate skewed text
        lang='en',
        use_gpu=False,        # Set True if GPU available
        show_log=False
    )

    def run_paddle_ocr(image_path: str) -> list:
        """
        Run PaddleOCR and return structured results.
        Returns: list of {text, confidence, bbox}
        """
        result = ocr.ocr(image_path, cls=True)
        lines = []
        for page in result:
            if page:
                for line in page:
                    bbox, (text, conf) = line
                    lines.append({
                        'text': text,
                        'confidence': round(conf, 4),
                        'bbox': bbox,
                        'x': int(bbox[0][0]),
                        'y': int(bbox[0][1])
                    })
        # Sort by reading order (top-to-bottom, left-to-right)
        lines.sort(key=lambda l: (l['y'], l['x']))
        return lines

    # Create a synthetic invoice image for testing
    def create_test_invoice_image(path='test_invoice.png'):
        img = Image.new('RGB', (600, 400), color='white')
        draw = ImageDraw.Draw(img)
        lines = [
            (50, 30,  'INVOICE',             24),
            (50, 70,  'Invoice No: INV-2024-0451', 16),
            (50, 100, 'Date: March 15, 2024',    16),
            (50, 140, 'Bill To: GlobalTech Ltd', 16),
            (50, 170, 'Address: Mumbai, India',  16),
            (50, 220, 'Description          Amount', 14),
            (50, 250, 'Software Development  Rs. 10,000', 14),
            (50, 280, 'Consulting Services   Rs.  2,450', 14),
            (50, 320, 'TOTAL:               Rs. 12,450', 16),
            (50, 360, 'GST @ 18%:           Rs.  2,241', 14),
        ]
        for x, y, text, size in lines:
            draw.text((x, y), text, fill='black')
        img.save(path)
        return path

    img_path = create_test_invoice_image()
    ocr_results = run_paddle_ocr(img_path)

    print('PaddleOCR Results from synthetic invoice:')
    print('=' * 60)
    full_text = []
    for line in ocr_results:
        print(f"  [{line['confidence']:.2%}] {line['text']}")
        full_text.append(line['text'])

    print(f'\n📝 Reconstructed text:\n{" ".join(full_text)}')

except ImportError:
    print('⚠️  PaddleOCR not installed. Run: pip install paddlepaddle paddleocr')
    print('\nSimulating OCR output structure:')
    ocr_results = [
        {'text': 'INVOICE', 'confidence': 0.9987, 'bbox': [[50,30],[150,30],[150,60],[50,60]], 'x': 50, 'y': 30},
        {'text': 'Invoice No: INV-2024-0451', 'confidence': 0.9912, 'bbox': [[50,70],[320,70],[320,95],[50,95]], 'x': 50, 'y': 70},
        {'text': 'Date: March 15, 2024', 'confidence': 0.9876, 'bbox': [[50,100],[280,100],[280,125],[50,125]], 'x': 50, 'y': 100},
        {'text': 'Bill To: GlobalTech Ltd', 'confidence': 0.9901, 'bbox': [[50,140],[300,140],[300,165],[50,165]], 'x': 50, 'y': 140},
        {'text': 'TOTAL: Rs. 12,450', 'confidence': 0.9845, 'bbox': [[50,320],[250,320],[250,345],[50,345]], 'x': 50, 'y': 320},
    ]
    for r in ocr_results:
        print(f"  [{r['confidence']:.2%}] {r['text']}")

In [ ]:
# ─── 4.2 AWS Textract Integration ───────────────────────────
import boto3
from botocore.exceptions import NoCredentialsError

def textract_extract_document(file_path: str, region: str = 'ap-south-1') -> dict:
    """
    Extract text, forms, and tables from a document using AWS Textract.

    For single-page: Uses synchronous detect_document_text (fast, <5 sec)
    For multi-page:  Use start_document_analysis (async, S3 required)
    """
    textract = boto3.client('textract', region_name=region)

    with open(file_path, 'rb') as f:
        doc_bytes = f.read()

    # ── Option 1: Raw text only (fastest) ──
    text_response = textract.detect_document_text(
        Document={'Bytes': doc_bytes}
    )

    # ── Option 2: Forms + Tables (richer, slower) ──
    analysis_response = textract.analyze_document(
        Document={'Bytes': doc_bytes},
        FeatureTypes=['FORMS', 'TABLES', 'QUERIES'],
        QueriesConfig={
            'Queries': [
                {'Text': 'What is the invoice number?', 'Alias': 'INVOICE_NUMBER'},
                {'Text': 'What is the total amount?',   'Alias': 'TOTAL_AMOUNT'},
                {'Text': 'What is the invoice date?',   'Alias': 'INVOICE_DATE'},
            ]
        }
    )

    # Parse blocks
    lines = [b['Text'] for b in text_response['Blocks'] if b['BlockType'] == 'LINE']

    # Parse key-value pairs from FORMS
    kv_pairs = {}
    for block in analysis_response['Blocks']:
        if block['BlockType'] == 'KEY_VALUE_SET' and 'KEY' in block.get('EntityTypes', []):
            key_text = block.get('Text', '')
            # Find associated VALUE block
            for rel in block.get('Relationships', []):
                if rel['Type'] == 'VALUE':
                    kv_pairs[key_text] = rel['Ids']  # Simplified

    return {
        'raw_text': '\n'.join(lines),
        'line_count': len(lines),
        'key_values': kv_pairs,
    }


# ── Simulated Textract response (for offline testing) ──
def simulate_textract_response(text: str) -> dict:
    """Simulates Textract output structure for development"""
    lines = text.split('.')
    blocks = [
        {'BlockType': 'LINE', 'Text': line.strip(), 'Confidence': 99.1, 'Id': str(i),
         'Geometry': {'BoundingBox': {'Width': 0.6, 'Height': 0.02, 'Left': 0.1, 'Top': i * 0.05}}}
        for i, line in enumerate(lines) if line.strip()
    ]
    return {
        'Blocks': blocks,
        'DocumentMetadata': {'Pages': 1},
        'ResponseMetadata': {'HTTPStatusCode': 200}
    }

# Test with simulated response
test_text = ALL_DOCS[0]['text']
sim_response = simulate_textract_response(test_text)

print('Simulated Textract Response Structure:')
print(f'  Total blocks: {len(sim_response["Blocks"])}')
print(f'  Pages: {sim_response["DocumentMetadata"]["Pages"]}')
print()
print('Extracted lines:')
for block in sim_response['Blocks']:
    conf = block.get('Confidence', 0)
    print(f"  [{conf:.1f}%] {block['Text']}")

print('\n💡 For real Textract: set AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY in env')

---
# 📓 PHASE 5: End-to-End NLP Pipeline
**Goal:** Chain OCR → text cleaning → spaCy NER → BERT classification → vector search  
**Duration:** Week 9–10

In [ ]:
# ─── 5.1 Full Document Intelligence Pipeline ─────────────────
import re
from dataclasses import dataclass, field
from typing import List, Optional

@dataclass
class DocumentResult:
    """Structured output from the NLP pipeline"""
    raw_text: str
    cleaned_text: str = ''
    doc_type: str = 'unknown'
    entities: List[dict] = field(default_factory=list)
    chunks: List[str] = field(default_factory=list)
    confidence: float = 0.0
    metadata: dict = field(default_factory=dict)


class DocumentIntelligencePipeline:
    """
    Stage 1: OCR (PaddleOCR / Textract / raw text)
    Stage 2: Text cleaning & normalization
    Stage 3: Document classification (rule-based or BERT)
    Stage 4: Named Entity Recognition (spaCy)
    Stage 5: Chunking for downstream embedding
    """

    def __init__(self, chunk_size: int = 200, chunk_overlap: int = 50):
        self.nlp = spacy.load('en_core_web_sm')
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

        # Document type rules
        self.doc_rules = {
            'invoice':   ['invoice', 'inv-', 'payment due', 'bill to', 'amount payable'],
            'contract':  ['agreement', 'hereby', 'terms and conditions', 'effective date'],
            'complaint': ['complaint', 'refund', 'damaged', 'disappointed', 'urgent'],
            'medical':   ['patient', 'hospital', 'admitted', 'diagnosis', 'discharged'],
            'news':      ['reported', 'quarter', 'profit', 'announced', 'growth'],
        }

    def stage1_ocr(self, input_data: str, source: str = 'text') -> str:
        """Stage 1: Get raw text (from OCR or direct input)"""
        if source == 'text':
            return input_data  # Already text
        elif source == 'paddle':
            # In production: return run_paddle_ocr(input_data)
            return f'[PaddleOCR output from {input_data}]'
        elif source == 'textract':
            # In production: return textract_extract_document(input_data)['raw_text']
            return f'[Textract output from {input_data}]'
        return input_data

    def stage2_clean(self, text: str) -> str:
        """Stage 2: Clean and normalize text"""
        text = re.sub(r'\s+', ' ', text)        # normalize whitespace
        text = re.sub(r'[^\x00-\x7F]+', ' ', text)  # remove non-ASCII (OCR artifacts)
        text = re.sub(r'(\w)- (\w)', r'\1\2', text)  # fix hyphenated line breaks
        text = text.strip()
        return text

    def stage3_classify(self, text: str) -> tuple:
        """Stage 3: Classify document type"""
        text_lower = text.lower()
        scores = {label: sum(1 for kw in kws if kw in text_lower)
                  for label, kws in self.doc_rules.items()}
        best = max(scores, key=scores.get)
        total = sum(scores.values()) or 1
        confidence = scores[best] / total
        return (best if scores[best] > 0 else 'unknown', confidence)

    def stage4_extract_entities(self, text: str) -> list:
        """Stage 4: Named Entity Recognition"""
        doc = self.nlp(text)
        return [
            {'text': ent.text, 'label': ent.label_,
             'start': ent.start_char, 'end': ent.end_char,
             'description': spacy.explain(ent.label_)}
            for ent in doc.ents
        ]

    def stage5_chunk(self, text: str) -> list:
        """Stage 5: Sliding window chunking for embedding"""
        words = text.split()
        chunks = []
        step = self.chunk_size - self.chunk_overlap
        for i in range(0, len(words), step):
            chunk = ' '.join(words[i:i + self.chunk_size])
            if len(chunk) > 20:
                chunks.append(chunk)
        return chunks

    def run(self, input_data: str, source: str = 'text') -> DocumentResult:
        """Execute full pipeline"""
        raw = self.stage1_ocr(input_data, source)
        cleaned = self.stage2_clean(raw)
        doc_type, confidence = self.stage3_classify(cleaned)
        entities = self.stage4_extract_entities(cleaned)
        chunks = self.stage5_chunk(cleaned)

        return DocumentResult(
            raw_text=raw,
            cleaned_text=cleaned,
            doc_type=doc_type,
            entities=entities,
            chunks=chunks,
            confidence=confidence,
            metadata={'char_count': len(cleaned), 'chunk_count': len(chunks)}
        )


# Run pipeline on all seed documents
pipeline = DocumentIntelligencePipeline(chunk_size=50, chunk_overlap=10)

print('Document Intelligence Pipeline Results:')
print('=' * 70)

for doc_data in ALL_DOCS[:5]:
    result = pipeline.run(doc_data['text'])
    correct = '✅' if result.doc_type == doc_data['label'] else '❌'
    print(f"{correct} [{result.doc_type}] (conf: {result.confidence:.0%})")
    print(f"   Text: {doc_data['text'][:60]}...")
    print(f"   Entities: {[(e['text'], e['label']) for e in result.entities[:3]]}")
    print(f"   Chunks: {len(result.chunks)} | Chars: {result.metadata['char_count']}")
    print()

In [ ]:
# ─── 5.2 Sentence Embeddings + FAISS Vector Store ───────────
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load embedding model
embedder = SentenceTransformer('all-MiniLM-L6-v2')  # 384-dim, fast

# Embed all seed documents
texts = [doc['text'] for doc in ALL_DOCS]
labels = [doc['label'] for doc in ALL_DOCS]

print('🔄 Generating embeddings...')
embeddings = embedder.encode(texts, show_progress_bar=True, normalize_embeddings=True)
print(f'Embedding matrix: {embeddings.shape}')

# Build FAISS index (cosine similarity via normalized L2)
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # Inner product = cosine for normalized vectors
index.add(embeddings.astype('float32'))
print(f'FAISS index: {index.ntotal} vectors, dim={dim}')

def semantic_search(query: str, top_k: int = 3) -> list:
    """Find most similar documents to a query"""
    q_emb = embedder.encode([query], normalize_embeddings=True)
    scores, indices = index.search(q_emb.astype('float32'), top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        results.append({
            'text': texts[idx][:80] + '...',
            'label': labels[idx],
            'similarity': float(score)
        })
    return results

# Test queries
queries = [
    'show me unpaid invoices',
    'hospital patient discharge information',
    'quarterly profit earnings report',
]

print('\nSemantic Search Results:')
print('=' * 70)
for q in queries:
    print(f'\n🔍 Query: "{q}"')
    for r in semantic_search(q, top_k=2):
        print(f"  [{r['similarity']:.3f}] ({r['label']}) {r['text']}")

---
# 📒 PHASE 6: Agentic AI Foundation
**Goal:** Build a document Q&A agent using Claude + tools + the NLP pipeline  
**Duration:** Week 11–12

In [ ]:
# ─── 6.1 Define Agent Tools ─────────────────────────────────
import json

# Tool definitions for Claude
AGENT_TOOLS = [
    {
        'name': 'search_documents',
        'description': 'Semantically search the document corpus. Use for questions about document content.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'query': {'type': 'string', 'description': 'Search query'},
                'top_k': {'type': 'integer', 'description': 'Number of results (default 3)', 'default': 3},
                'filter_label': {'type': 'string', 'description': 'Filter by document type (invoice/contract/complaint/medical/news)', 'default': None}
            },
            'required': ['query']
        }
    },
    {
        'name': 'extract_entities',
        'description': 'Extract named entities (persons, organizations, dates, money) from a text.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'text': {'type': 'string', 'description': 'Text to extract entities from'},
                'entity_types': {'type': 'array', 'items': {'type': 'string'}, 'description': 'Filter by type: PERSON, ORG, DATE, MONEY, GPE'}
            },
            'required': ['text']
        }
    },
    {
        'name': 'classify_document',
        'description': 'Classify what type of document a text belongs to.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'text': {'type': 'string', 'description': 'Document text to classify'}
            },
            'required': ['text']
        }
    },
    {
        'name': 'summarize_with_claude',
        'description': 'Summarize a set of documents into a coherent answer.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'documents': {'type': 'array', 'items': {'type': 'string'}, 'description': 'Documents to summarize'},
                'instruction': {'type': 'string', 'description': 'What to focus on in the summary'}
            },
            'required': ['documents', 'instruction']
        }
    }
]

print('✅ Agent tools defined:')
for tool in AGENT_TOOLS:
    print(f"  🔧 {tool['name']}: {tool['description'][:60]}...")

In [ ]:
# ─── 6.2 Tool Executor ───────────────────────────────────────
nlp_for_agent = spacy.load('en_core_web_sm')
pipeline_for_agent = DocumentIntelligencePipeline()

def execute_tool(tool_name: str, tool_input: dict) -> str:
    """
    Execute agent tool calls and return string results.
    This is the bridge between Claude's tool calls and our NLP stack.
    """

    if tool_name == 'search_documents':
        query = tool_input['query']
        top_k = tool_input.get('top_k', 3)
        filter_label = tool_input.get('filter_label')
        results = semantic_search(query, top_k=top_k * 2)  # Fetch extra for filtering
        if filter_label:
            results = [r for r in results if r['label'] == filter_label]
        results = results[:top_k]
        if not results:
            return 'No documents found matching the query.'
        output = []
        for i, r in enumerate(results):
            output.append(f"[Doc {i+1}] Type: {r['label']} | Similarity: {r['similarity']:.2%}\n{r['text']}")
        return '\n\n'.join(output)

    elif tool_name == 'extract_entities':
        text = tool_input['text']
        entity_types = tool_input.get('entity_types', [])
        doc = nlp_for_agent(text)
        entities = [(e.text, e.label_) for e in doc.ents
                    if not entity_types or e.label_ in entity_types]
        if not entities:
            return 'No entities found.'
        return json.dumps([{'text': t, 'type': l} for t, l in entities], indent=2)

    elif tool_name == 'classify_document':
        text = tool_input['text']
        result = pipeline_for_agent.run(text)
        return f"Document type: {result.doc_type} (confidence: {result.confidence:.0%})"

    elif tool_name == 'summarize_with_claude':
        documents = tool_input['documents']
        instruction = tool_input['instruction']
        context = '\n\n'.join([f'Document {i+1}: {d}' for i, d in enumerate(documents)])
        response = client.messages.create(
            model='claude-sonnet-4-20250514',
            max_tokens=500,
            messages=[{
                'role': 'user',
                'content': f'Based on these documents:\n\n{context}\n\n{instruction}'
            }]
        )
        return response.content[0].text

    return f'Unknown tool: {tool_name}'

print('✅ Tool executor ready')

In [ ]:
# ─── 6.3 Agent ReAct Loop ────────────────────────────────────
def run_document_agent(user_query: str, max_turns: int = 5, verbose: bool = True) -> str:
    """
    ReAct agent loop:
    1. Claude receives user query + available tools
    2. Claude decides which tool to call
    3. We execute the tool and return results
    4. Claude synthesizes final answer

    Loops until Claude stops calling tools (max_turns safety limit).
    """
    system_prompt = """
You are a Document Intelligence Agent specialized in analyzing business documents.
You have access to a corpus of invoices, contracts, complaints, medical notes, and news articles.

When answering questions:
1. Use search_documents to find relevant documents first
2. Use extract_entities to pull out specific facts (dates, amounts, names)
3. Use classify_document if document type is unclear
4. Always cite which document your answer comes from
5. If information is not in the corpus, say so clearly
"""

    messages = [{'role': 'user', 'content': user_query}]

    if verbose:
        print(f'\n🤖 Agent Query: "{user_query}"')
        print('=' * 60)

    for turn in range(max_turns):
        response = client.messages.create(
            model='claude-sonnet-4-20250514',
            max_tokens=1000,
            system=system_prompt,
            tools=AGENT_TOOLS,
            messages=messages
        )

        # Append assistant response to history
        messages.append({'role': 'assistant', 'content': response.content})

        # Check for tool calls
        tool_calls = [b for b in response.content if b.type == 'tool_use']

        if not tool_calls:
            # No more tool calls → final answer
            final_answer = ' '.join(b.text for b in response.content if hasattr(b, 'text'))
            if verbose:
                print(f'\n💬 Final Answer:\n{final_answer}')
            return final_answer

        # Execute all tool calls
        tool_results = []
        for tc in tool_calls:
            if verbose:
                print(f'  🔧 Tool: {tc.name}({json.dumps(tc.input)[:80]}...)')
            result = execute_tool(tc.name, tc.input)
            if verbose:
                print(f'  📤 Result: {result[:100]}...')
            tool_results.append({
                'type': 'tool_result',
                'tool_use_id': tc.id,
                'content': result
            })

        messages.append({'role': 'user', 'content': tool_results})

    return 'Max turns reached without final answer.'


# ── Test the Agent ──
queries = [
    'What was the total invoice amount for GlobalTech and when was it due?',
    'Which organizations are mentioned across our medical and legal documents?',
    'Summarize all the financial information you can find in the corpus.',
]

for query in queries[:1]:  # Run one to avoid API costs — remove slice for all
    answer = run_document_agent(query, verbose=True)

---
# 🏆 CAPSTONE: Full Document Q&A System
**Exercise:** Wire everything together into a function that accepts any document file (PDF/image/text) and answers questions about it.

```python
def document_qa_system(file_path: str, question: str, ocr_engine: str = 'paddle') -> str:
    # Stage 1: OCR the document
    raw_text = pipeline.stage1_ocr(file_path, source=ocr_engine)

    # Stage 2-5: Run NLP pipeline
    result = pipeline.run(raw_text)

    # Add to vector index
    embedding = embedder.encode([result.cleaned_text], normalize_embeddings=True)
    index.add(embedding.astype('float32'))
    texts.append(result.cleaned_text)
    labels.append(result.doc_type)

    # Run agent Q&A
    return run_document_agent(question)
```

## 📝 Exercises per Phase

| Phase | Exercise |
|-------|----------|
| 1 | Clean and vectorize 50 real emails from your domain using TF-IDF |
| 2 | Add 10 custom EntityRuler patterns for your domain (GST, PAN, bank codes) |
| 3 | Fine-tune DistilBERT on 100 labelled documents from your corpus |
| 4 | Process 10 real PDF invoices with PaddleOCR and measure accuracy |
| 5 | Build the full pipeline and index 100 documents in FAISS |
| 6 | Add a `get_all_invoices_above` tool to the agent and test it |

---
*Generated with Anthropic Claude API · NLP Agentic AI Training · 2024*